In [2]:
import numpy as np

class SimpleGridWorld:
    def __init__(self, size=5):
        self.size = size
        self.agent_pos = [0, 0]
        self.goal_pos = [size - 1, size - 1]
        self.trap_pos = [size // 2, size // 2]

    def reset(self):
        self.agent_pos = [0, 0]
        return tuple(self.agent_pos)

    def step(self, action):
        # Actions: 0: Up, 1: Down, 2: Left, 3: Right
        if action == 0: # Up
            self.agent_pos[0] = max(0, self.agent_pos[0] - 1)
        elif action == 1: # Down
            self.agent_pos[0] = min(self.size - 1, self.agent_pos[0] + 1)
        elif action == 2: # Left
            self.agent_pos[1] = max(0, self.agent_pos[1] - 1)
        elif action == 3: # Right
            self.agent_pos[1] = min(self.size - 1, self.agent_pos[1] + 1)

        reward = -0.1 # Small penalty for each step to encourage efficiency
        done = False

        if self.agent_pos == self.goal_pos:
            reward = 1.0 # Big positive reward for reaching the goal
            done = True
        elif self.agent_pos == self.trap_pos:
            reward = -1.0 # Big negative reward for hitting a trap
            done = True

        return tuple(self.agent_pos), reward, done, {}

    def render(self):
        grid = np.full((self.size, self.size), '.')
        grid[self.goal_pos[0], self.goal_pos[1]] = 'G'
        grid[self.trap_pos[0], self.trap_pos[1]] = 'T'
        grid[self.agent_pos[0], self.agent_pos[1]] = 'A'
        for row in grid:
            print(' '.join(row))
        print("\n")

print("SimpleGridWorld environment defined.")
env = SimpleGridWorld(size=5)
state = env.reset()
print("Initial State:")
env.render()

done = False
total_reward = 0

actions_to_try = [1, 1, 3, 2, 1, 1,3,3,3,3]

for i, action in enumerate(actions_to_try):
    if done:
        print("Episode ended prematurely.")
        break
    print(f"Taking action: {action}")
    next_state, reward, done, _ = env.step(action)
    total_reward += reward
    print(f"State: {next_state}, Reward: {reward}, Done: {done}")
    env.render()

    if done:
        print(f"Episode finished! Total Reward: {total_reward:.2f}")
        break

if not done:
    print(f"Episode did not finish. Total Reward: {total_reward:.2f}")

SimpleGridWorld environment defined.
Initial State:
A . . . .
. . . . .
. . T . .
. . . . .
. . . . G


Taking action: 1
State: (1, 0), Reward: -0.1, Done: False
. . . . .
A . . . .
. . T . .
. . . . .
. . . . G


Taking action: 1
State: (2, 0), Reward: -0.1, Done: False
. . . . .
. . . . .
A . T . .
. . . . .
. . . . G


Taking action: 3
State: (2, 1), Reward: -0.1, Done: False
. . . . .
. . . . .
. A T . .
. . . . .
. . . . G


Taking action: 2
State: (2, 0), Reward: -0.1, Done: False
. . . . .
. . . . .
A . T . .
. . . . .
. . . . G


Taking action: 1
State: (3, 0), Reward: -0.1, Done: False
. . . . .
. . . . .
. . T . .
A . . . .
. . . . G


Taking action: 1
State: (4, 0), Reward: -0.1, Done: False
. . . . .
. . . . .
. . T . .
. . . . .
A . . . G


Taking action: 3
State: (4, 1), Reward: -0.1, Done: False
. . . . .
. . . . .
. . T . .
. . . . .
. A . . G


Taking action: 3
State: (4, 2), Reward: -0.1, Done: False
. . . . .
. . . . .
. . T . .
. . . . .
. . A . G


Taking action: 3

In [4]:
import numpy as np

class RecyclingRobotBandit:
    def __init__(self, num_actions=3, epsilon=0.1):
        """
        Initializes the Recycling Robot simulation.
        Actions:
        0: Intensely search for recyclables (High reward risk, higher energy cost)
        1: Passively search for recyclables (Medium reward, low energy cost)
        2: Go back to the docking station to clear bins and optimize components (No reward, system reset)
        """
        self.num_actions = num_actions
        self.epsilon = epsilon

        self.action_counts = np.zeros(num_actions)
        self.action_rewards = np.zeros(num_actions)
        self.total_collected_items = 0

        self.true_success_rates = {
            0: 0.75,
            1: 0.45,
            2: 0.05
        }

    def select_action(self):
        """Selects an action using the Epsilon-Greedy Policy."""
        if np.random.rand() < self.epsilon:
            return np.random.randint(0, self.num_actions)
        else:
            average_rewards = [
                self.action_rewards[i] / self.action_counts[i] if self.action_counts[i] > 0 else 0
                for i in range(self.num_actions)
            ]
            return np.argmax(average_rewards)

    def simulate_environment(self, action):
        """Simulates environment feedback (1 if item collected, 0 if empty space found)."""
        success_probability = self.true_success_rates[action]
        return 1 if np.random.rand() < success_probability else 0

    def run_simulation(self, iterations=1000):
        """Runs the operational simulation loop for a set number of steps."""
        for step in range(iterations):
            action = self.select_action()
            reward = self.simulate_environment(action)

            self.action_counts[action] += 1
            self.action_rewards[action] += reward
            self.total_collected_items += reward

    def display_metrics(self):
        """Outputs performance metrics of the recycling operation."""
        print("--- Recycling Robot Operations Summary ---")
        print(f"Total Steps Simulated: {int(sum(self.action_counts))}")
        print(f"Total Recyclable Items Collected: {self.total_collected_items}")
        print("\nBehavior Performance Breakdown:")

        actions_labels = {0: "Intense Search", 1: "Passive Search", 2: "Docking / Reset"}
        for i in range(self.num_actions):
            avg = self.action_rewards[i] / self.action_counts[i] if self.action_counts[i] > 0 else 0
            print(f"[{actions_labels[i]}] -> Times Chosen: {int(self.action_counts[i])}, Items Found: {int(self.action_rewards[i])}, Success Rate: {avg:.4f}")

if __name__ == "__main__":
    np.random.seed(42)

    robot = RecyclingRobotBandit(epsilon=0.1)

    robot.run_simulation(iterations=1000)
    robot.display_metrics()


--- Recycling Robot Operations Summary ---
Total Steps Simulated: 1000
Total Recyclable Items Collected: 711

Behavior Performance Breakdown:
[Intense Search] -> Times Chosen: 929, Items Found: 691, Success Rate: 0.7438
[Passive Search] -> Times Chosen: 41, Items Found: 19, Success Rate: 0.4634
[Docking / Reset] -> Times Chosen: 30, Items Found: 1, Success Rate: 0.0333
